# GSN TPM Analytics Notebook
## Google Global Submarine Networks — Program Intelligence
**Author:** Nithyashree Babu | Technical Program Manager Candidate  
**Role Target:** Google GSN TPM — EMEA Submarine Cable Infrastructure

---
This notebook demonstrates end-to-end analytical competency across:
- Subsea infrastructure milestone tracking & critical path identification
- Earned Value Management (EVM) for multi-project portfolios
- Statistical fiber health monitoring (KS-Test, PSI)
- Risk scoring, Pareto analysis & EMV calculation
- Executive-ready insights & visualizations

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sys, os
sys.path.insert(0, '../src')
from analytics import *

# Load all datasets
DATA = '../data'
projects   = pd.read_csv(f'{DATA}/projects.csv')
milestones = pd.read_csv(f'{DATA}/milestones.csv')
risks      = pd.read_csv(f'{DATA}/risks_scored.csv')
kpis       = pd.read_csv(f'{DATA}/kpi_timeseries.csv')
fiber      = pd.read_csv(f'{DATA}/fiber_health.csv')
evm        = pd.read_csv(f'{DATA}/evm_metrics.csv')
cp         = pd.read_csv(f'{DATA}/critical_path.csv')
fiber_anom = pd.read_csv(f'{DATA}/fiber_anomalies.csv')
bottleneck = pd.read_csv(f'{DATA}/bottlenecks.csv')

print(f'Loaded {len(projects)} cable systems | {len(milestones)} milestones | {len(risks)} risks')

## 1. Portfolio Health — Executive Summary KPIs

In [ ]:
summary = projects.agg({
    'budget_m_usd': 'sum',
    'spend_to_date_m_usd': 'sum',
    'delay_days': ['mean', 'max', 'sum'],
    'completion_pct': 'mean'
})
print('Portfolio Summary:')
print(f'  Total Budget:        ${projects.budget_m_usd.sum():.1f}M')
print(f'  Total Spend:         ${projects.spend_to_date_m_usd.sum():.1f}M')
print(f'  Avg Completion:      {projects.completion_pct.mean():.1f}%')
print(f'  On-Track Projects:   {(projects.status=="On Track").sum()}/{len(projects)}')
print(f'  Total Cable km:      {projects.length_km.sum():,} km')
print(f'  Avg Schedule Delay:  {projects.delay_days.mean():.0f} days')

## 2. Critical Path Analysis

In [ ]:
print('Top Bottleneck Milestones (Critical Path):')
print(bottleneck[['milestone','avg_delay','max_delay','projects_affected','bottleneck_score']].head(7).to_string(index=False))

fig = px.bar(bottleneck.head(8), x='bottleneck_score', y='milestone', orientation='h',
             color='avg_delay', color_continuous_scale='RdYlGn_r',
             title='Bottleneck Score by Critical Path Milestone')
fig.show()

## 3. Earned Value Management (EVM)

In [ ]:
print('EVM Metrics by Project:')
print(evm[['cable_name','spi','cpi','budget_m','eac_m','vac_m','evm_health']].to_string(index=False))

# SPI/CPI quadrant
fig = px.scatter(evm, x='spi', y='cpi', text='cable_name',
                 color='evm_health', size='budget_m',
                 color_discrete_map={'Green':'#34a853','Yellow':'#fbbc04','Red':'#ea4335'},
                 title='SPI vs CPI — EVM Quadrant Analysis')
fig.add_vline(x=1.0, line_dash='dash')
fig.add_hline(y=1.0, line_dash='dash')
fig.update_traces(textposition='top center')
fig.show()

## 4. Risk Intelligence — Pareto & EMV

In [ ]:
top_risks = risks.sort_values('composite_score', ascending=False).head(10)
print('Top 10 Risks by Composite Score:')
print(top_risks[['risk_id','cable_name','category','severity','composite_score','emv_m']].to_string(index=False))

fig = px.scatter(risks[risks['status']=='Open'], x='probability', y='impact',
                 color='severity', size='composite_score',
                 color_discrete_map={'Critical':'#ea4335','High':'#fbbc04','Medium':'#4285f4','Low':'#34a853'},
                 title='Risk Register Heatmap')
fig.show()

## 5. Fiber Health — Statistical Anomaly Detection (KS-Test & PSI)

In [ ]:
print('Fiber Anomaly Detection Results:')
print(fiber_anom[['cable_name','ks_statistic','ks_pvalue','psi_osnr','anomaly_rate_pct','health_score']].to_string(index=False))
print('\nInterpretation:')
print('  KS p-value < 0.05 → Distribution shift detected')
print('  PSI > 0.20       → Significant feature drift (alert)')
print('  Health Score     → 100 - anomaly_rate%')

## 6. Standardization — Global Process Consistency Metrics

In [ ]:
# Simulate standardization metric across regions
standardization_df = pd.DataFrame({
    'region': ['EMEA-UK','EMEA-EU','EMEA-Africa','APAC-India','APAC-SEA'],
    'reporting_template_compliance': [0.97, 0.92, 0.85, 0.88, 0.91],
    'tooling_adoption': [0.95, 0.90, 0.78, 0.82, 0.88],
    'milestone_def_consistency': [0.99, 0.95, 0.80, 0.85, 0.92],
})
standardization_df['overall_score'] = standardization_df[[
    'reporting_template_compliance','tooling_adoption','milestone_def_consistency'
]].mean(axis=1).round(3)

print('Global Standardization Scorecard:')
print(standardization_df.to_string(index=False))

fig = px.bar(standardization_df.melt(id_vars='region', var_name='metric', value_name='score'),
             x='region', y='score', color='metric', barmode='group',
             title='Global Process Standardization by Region')
fig.update_yaxes(range=[0.6, 1.0])
fig.show()